<a href="https://colab.research.google.com/github/CIS5190/cis-4190-5190-fall-26/blob/main/hw3/hw3_skeleton.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CIS 4190/5190 Fall 2026 - Homework 3**

# **Tips for Coding Homeworks**
Dear class, here are some tips to follow in coding HWs in this class, to minimize the chances of hitting hard-to-resolve bugs:

1.   **Do not set/alter the random seeds we provide.** (Seeds fixed in setup or "don't change this line" cells exist so that everyone's numbers, and the hidden autograder tests, match up.)
2.   Most bugs come down to minor errors in your code. The hardest and most important part of coding is debugging; debugging ML code is a key skillset you should acquire in this class.
3.   Good python coding avoids loops if at all possible: in particular, **for any vector and matrix operations, use `numpy` functions instead of loops**. This will often speed up your code by orders of magnitude. You may also want to take a look at the [python primer](https://www.engineering.upenn.edu/~cis5190/fall2026/docs/basics_python_numpy.ipynb).
4.   Pay **extra attention to "copying" vs "moving" objects** i.e. take care during assignment operations. Otherwise, it may lead to incorrect results or Out Of Memory issues and Colab notebook restarts.
5.   **Your `KMeans` class is graded on its own, with numpy.** The autograder takes the class you wrote and runs it by itself, where only `np`, `pd`, `math` and `random` exist — no scipy, no scikit-learn, and nothing from your other cells. So write `KMeans` with numpy, and keep any helper it needs as a method of the class. (An `import` at the top of the class body does not help: methods cannot see names defined there.) Everything else in this homework is graded from the numbers your notebook computes, so this rule applies only to `KMeans`.
6.   **Part 3 trains much faster on a GPU.** It runs on a CPU too, just more slowly. To use one in Colab, choose *Runtime → Change runtime type → T4 GPU* before you start Part 3. Changing the runtime restarts the notebook, so afterwards re-run every cell from the top down to *Initialize the Autograder*.

Have fun!

> **Important: Save a Copy Before Starting**
> Please click *"Copy to Drive"* in the top menu bar or go to *File → Save a Copy in Drive*.
> This shared copy is read-only, so any edits you make to it will not be saved. Work in your
> own copy.

---

> **Points**
> This assignment is worth **50 points**: 43 points that every student completes, plus 7
> points marked **[5190 Extra]**, which are section 1.6 (feature scaling and K-means, 3 pts)
> and section 2.3 (K-means on the PCA plane, 4 pts).
>
> - **CIS 4190** is graded out of **43**. The [5190 Extra] parts are optional for you.
> - **CIS 5190** is graded out of **50**. The [5190 Extra] parts are required for you.
>
> CIS 4190 students are welcome to attempt the [5190 Extra] parts for credit, but your total
> for this assignment is still capped at 43 points.
>
> Section 4 (an LLM as the classifier) is optional and **not graded**. It only runs on your own
> computer, on PennNet; Colab cannot reach the course LLM.

---

> **How this notebook is graded**
> 35 of the 50 points are graded by the autograder in this notebook, from the
> PennGrader cells. The other 15 are graded by hand from the plots and written answers
> you leave in the notebook — sections 1.4, 1.5, 1.6, 2.1 (the plot and comment), 2.2.2 and 2.3.
> **Download the notebook with its outputs intact**, or there will be nothing there to mark.
>
> **Your most recent submission for each grading cell is the one that counts** — re-running a
> grading cell with a worse answer will lower your recorded score. You may resubmit each
> grading cell up to 100 times per day.
>
> **When you submit to Gradescope, it re-runs your notebook from the top, in order**, and every
> grading cell not wrapped in `if NOTEBOOK:` submits again. Since your latest submission counts,
> what Gradescope records for those cells is what your notebook produces when run top to bottom.
> So if you edited code after running its grading cell, or ran cells out of order, re-run Parts 1
> and 2 and section 3.1 in order before you download. An error stops the re-run at that cell.
>
> Cells wrapped in `if NOTEBOOK:` are skipped there — Part 3's image download, its training and
> 3.5's grading cell among them — so **run 3.5's grading cell yourself, in Colab**.

## Setup Cell 1: Imports and Configuration

This cell imports the libraries the first two parts use. Part 3 imports PyTorch separately.

In [ ]:
import os
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)  # don't change this line!!
random.seed(42)     # don't change this line!!

## Setup Cell 2: Environment Detection

This cell detects whether the notebook is running in Google Colab or in the autograder.

- `NOTEBOOK` will be `True` if running in Colab and `False` in the autograder.
- Cells that start with `if NOTEBOOK:` are skipped by the autograder. Leave those guards in place.
- Do **not** modify this cell.

In [ ]:
# For autograder only, do not modify this cell.
# Determines if code is running in Google Colab or autograder environment

NOTEBOOK = (os.getenv('IS_AUTOGRADER') is None)

if NOTEBOOK:
    print("[INFO, OK] Google Colab.")
else:
    print("[INFO, OK] Autograder.")

## PennGrader Setup

As in the previous homeworks, you'll use PennGrader to get instant feedback on your answers.

Because section 1.2 asks you to submit an entire **class** (not just a single function), we need the
helper cell below (`grader_serialize`) that knows how to pull a full class definition out of
your notebook so PennGrader can grade it.

In [ ]:
%%capture
!pip3 install penngrader-client==0.5.13

In [ ]:
from penngrader.grader import *

### Autograder helper

This cell lets PennGrader read the source of a **class** you define. Do **not** modify this cell.

In [ ]:
# For autograder only, do not modify this cell.
# Lets PennGrader retrieve the full source of a class defined in an earlier cell.

import inspect, sys
from IPython.core.magics.code import extract_symbols

def new_getfile(object, _old_getfile=inspect.getfile):
    if not inspect.isclass(object):
        return _old_getfile(object)

    # Lookup by parent module (as in current inspect)
    if hasattr(object, '__module__'):
        object_ = sys.modules.get(object.__module__)
        if hasattr(object_, '__file__'):
            return object_.__file__

    # If parent module is __main__, lookup by methods
    for name, member in inspect.getmembers(object):
        if inspect.isfunction(member) and object.__qualname__ + '.' + member.__name__ == member.__qualname__:
            return inspect.getfile(member)
    else:
        raise TypeError('Source for {!r} not found'.format(object))
inspect.getfile = new_getfile

def grader_serialize(obj):
    cell_code = "".join(inspect.linecache.getlines(new_getfile(obj)))
    # When Gradescope re-runs the notebook it is one file, which can hold several
    # `class KMeans` blocks (say, the skeleton's and your finished one). Take the
    # definition this class actually came from: the one containing its own methods.
    import ast
    own = [m.__code__.co_firstlineno for _, m in inspect.getmembers(obj, inspect.isfunction)
           if m.__qualname__.startswith(obj.__qualname__ + ".")]
    try:
        tree = ast.parse(cell_code)
    except SyntaxError:
        tree = None
    if tree is not None:
        lines = cell_code.splitlines(keepends=True)
        for node in ast.walk(tree):
            if (isinstance(node, ast.ClassDef) and node.name == obj.__name__
                    and any(node.lineno <= l <= node.end_lineno for l in own)):
                start = min([node.lineno] + [d.lineno for d in node.decorator_list])
                return "".join(lines[start - 1:node.end_lineno])
    class_code = extract_symbols(cell_code, obj.__name__)[0][0]
    return class_code

### Autograder Setup

Please enter your **8-digit PennID** in the cell below:

In [ ]:
# PLEASE ENSURE YOUR PENN-ID IS ENTERED CORRECTLY.
# IF NOT, THE AUTOGRADER WON'T KNOW WHO TO ASSIGN POINTS TO IN OUR BACKEND.

STUDENT_ID = 12345678          # YOUR PENN-ID GOES HERE AS AN INTEGER

## Initialize the Autograder

Run the following cell to initialize the autograder. This autograder will let you submit your code directly from this notebook and immediately receive a score.

**NOTE:** We store your submissions and check them against other students' submissions. So, not that you would, but please do **not** cheat.

In [ ]:
%%writefile notebook-config.yaml

grader_api_url: "https://yj0vsrf8l4.execute-api.us-east-1.amazonaws.com/Initial/TestSandbox"
grader_api_key: "dBB5ucuV5L6GyKFKmlrfP4bKkhtnA0bP42lR3rSh"
token_generator_url: "https://yj0vsrf8l4.execute-api.us-east-1.amazonaws.com/Initial/GetTokens"
token_generator_api_key: "dBB5ucuV5L6GyKFKmlrfP4bKkhtnA0bP42lR3rSh"


In [ ]:
COURSE_NAME = 'cis5190_fall2026'
HOMEWORK_NUMBER = 3   # -> homework ID cis5190_fall2026_HW3

grader = PennGrader('notebook-config.yaml', HOMEWORK_NUMBER, STUDENT_ID, STUDENT_ID, COURSE_NAME)

# **1. K-Means Clustering [19 pts — 16 for CIS 4190]**

We will implement K-means clustering and run it on the Breast Cancer Wisconsin dataset. As
with all unsupervised learning, the goal is to discover structure in **unlabeled** data: we
are given $n$ samples $x_1, \dots, x_n \in \mathbb{R}^d$ and a number of clusters $K$, and we
want to split the samples into $K$ groups of similar points.

We use the notation from lecture. K-means keeps one **mean** (centroid) $\mu_j \in \mathbb{R}^d$
per cluster, $j = 1, \dots, K$, and assigns each sample to the cluster whose mean is nearest:

$$f_\mu(x) = \arg\min_j \lVert x - \mu_j \rVert_2 .$$

It chooses the means to minimize the **within-cluster sum of squared distances (WCSS)**:

$$L(\mu; Z) = \sum_{i=1}^{n} \lVert x_i - \mu_{f_\mu(x_i)} \rVert_2^2 .$$

The optimizer is **alternating minimization**: start from an initial set of means, then repeat
(1) assign every sample to its nearest mean, and (2) recompute each mean as the average of the
samples assigned to it — until the means stop changing.

In the code, $K$ is `n_clusters`, the means are the rows of a `(K, d)` array called `centroids`,
and the assignments $f_\mu(x_i)$ are a length-$n$ integer array called `clusters`, holding a
cluster id in $\{0, \dots, K-1\}$ for each sample.

## **1.1. Load the Dataset [ungraded]**

The data is a matrix `X` of shape $(n, d) = (569, 30)$: 569 tumour samples, each described by 30
measurements of cell nuclei (radius, texture, area, smoothness, ...). The labels `y`
(0 = malignant, 1 = benign) are **not** used by K-means; they come back in Part 2.

In [ ]:
from sklearn.datasets import load_breast_cancer

cancer_dataset = load_breast_cancer()
X = cancer_dataset.data                       # shape (569, 30), the raw features
y = cancer_dataset.target                     # 0 = malignant, 1 = benign
feature_names = list(cancer_dataset.feature_names)

print("X:", X.shape, " y:", y.shape)

## **1.2. Implement K-Means [10 pts, Autograded]**

Complete the `KMeans` class below. Four methods are yours, and each is graded separately:

| Method | Points | What it does |
|---|---|---|
| `compute_centroids(X, clusters)` | 2 | Returns a `(K, d)` array whose row `k` is the mean of the samples with `clusters == k`. You may assume every cluster has at least one sample; `fit` deals with empty ones. |
| `compute_distance(X, centroids)` | 2 | Returns an `(n, K)` array whose entry `[i, k]` is the **Euclidean distance** $\lVert x_i - \mu_k \rVert_2$ — the distance itself, not its square. |
| `find_closest_cluster(dist)` | 2 | Returns a flat integer array of shape `(n,)`: for each sample, the index of its nearest centroid. Use `np.argmin`, so a tie goes to the lower index. |
| `fit(X, init_centroids=None)` | 4 | Runs alternating minimization and returns `(clusters, centroids, iters_taken)`. |

**What `fit` must do**

1. **Initialize.** If `init_centroids` is given, start from it. Otherwise call the provided
   `self.init_centroids(X)`, which picks `K` distinct samples at random using `random_state`.
   That code is written for you so that everyone's K-means starts from the same place — **do not
   change it**. Section 1.3's autograded answer depends on it.
2. **Repeat** at most `max_iter` times:
   - assign every sample to its nearest centroid (`compute_distance`, then `find_closest_cluster`);
   - recompute the centroids (`compute_centroids`);
   - **if a cluster received no samples, keep its previous centroid** for that cluster. (Its mean
     is undefined: `np.mean` of nothing is `NaN`, and moving it to the origin is arbitrary.)
   - **stop** as soon as the centroids do not change from one iteration to the next (exact
     equality or `np.allclose` — either is fine).
3. **Return** `clusters` (shape `(n,)`), `centroids` (shape `(K, d)`), and `iters_taken`, the
   number of times your loop ran. `iters_taken` is for your own diagnostics: any whole number from
   1 to `max_iter` is accepted.

Write the class with numpy only (Tip 5). The four graded methods are tested on small fixtures
with known answers, including one where a cluster starts out empty.

In [ ]:
class KMeans:
    '''K-means clustering by alternating minimization.'''

    def __init__(self, n_clusters, max_iter=1000, random_state=0):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.random_state = random_state

    def init_centroids(self, X):
        """
        Provided -- do not change. Picks n_clusters distinct rows of X as the starting
        centroids, using numpy's RandomState seeded with random_state.

        INPUT:
        X - (n, d) array

        OUTPUT:
        centroids - (K, d) array, a copy of K rows of X
        """
        rng = np.random.RandomState(self.random_state)
        idx = rng.choice(X.shape[0], size=self.n_clusters, replace=False)
        return X[idx].astype(float)

    def compute_centroids(self, X, clusters):
        """
        Computes new centroid positions given the cluster assignments.

        INPUT:
        X - (n, d) array of samples
        clusters - (n,) integer array; clusters[i] is the cluster id of X[i]

        OUTPUT:
        centroids - (K, d) array; row k is the mean of the samples in cluster k
        """
        # STUDENT TODO START:
        raise NotImplementedError("1.2: compute_centroids")
        # STUDENT TODO END
        return centroids

    def compute_distance(self, X, centroids):
        """
        Computes the Euclidean distance from every sample to every centroid.

        INPUT:
        X - (n, d) array of samples
        centroids - (K, d) array

        OUTPUT:
        dist - (n, K) array; dist[i, k] is the Euclidean distance between X[i] and centroids[k]
        """
        # STUDENT TODO START:
        raise NotImplementedError("1.2: compute_distance")
        # STUDENT TODO END
        return dist

    def find_closest_cluster(self, dist):
        """
        Finds the nearest centroid for every sample.

        INPUT:
        dist - (n, K) array of distances, as returned by compute_distance

        OUTPUT:
        clusters - (n,) integer array; clusters[i] is the index of the centroid nearest X[i]
        """
        # STUDENT TODO START:
        raise NotImplementedError("1.2: find_closest_cluster")
        # STUDENT TODO END
        return clusters

    def fit(self, X, init_centroids=None):
        """
        Fits K-means to X by alternating minimization.

        INPUT:
        X - (n, d) array of samples
        init_centroids (optional) - (K, d) array of starting centroids. If None, use
                                    self.init_centroids(X).

        OUTPUT:
        clusters - (n,) integer array of cluster ids, from the last assignment step
        centroids - (K, d) array of final centroids
        iters_taken - number of iterations your loop ran (between 1 and max_iter)
        """
        if init_centroids is None:
            centroids = self.init_centroids(X)
        else:
            centroids = np.array(init_centroids, dtype=float)

        # STUDENT TODO START:
        # Repeat at most self.max_iter times:
        #   - assign each sample to its nearest centroid
        #   - recompute the centroids; an empty cluster keeps its previous centroid
        #   - stop when the centroids no longer change
        # Leave the results in `clusters`, `centroids` and `iters_taken`: the lines after
        # this block store and return them.
        raise NotImplementedError("1.2: fit")
        # STUDENT TODO END

        self.clusters = clusters
        self.centroids = centroids
        return self.clusters, self.centroids, iters_taken

### Check each method, then submit the class

Each local test below checks the same property as the hidden test of the same name, on a
smaller example. The hidden tests use different data, so passing locally is a good sign rather
than a guarantee.

In [ ]:
# Local test: compute_centroids. Row k is the mean of the samples in cluster k.
def test_compute_centroids():
    X_small = np.array([[0., 0.], [10., 10.], [2., 0.], [4., 6.], [12., 10.], [6., 6.]])
    clusters_small = np.array([0, 1, 0, 2, 1, 2])
    got = np.asarray(KMeans(3).compute_centroids(X_small, clusters_small), dtype=float)
    assert got.shape == (3, 2), f"expected shape (3, 2), got {got.shape}"
    assert np.allclose(got, [[1., 0.], [11., 10.], [5., 6.]]), f"wrong centroids:\n{got}"
    print("compute_centroids: local test passed")

test_compute_centroids()

In [ ]:
# PennGrader Grading Cell
grader.grade(test_case_id='test_compute_centroids', answer=grader_serialize(KMeans))

In [ ]:
# Local test: compute_distance. Euclidean distances (not squared), shape (n, K).
def test_distance():
    X_small = np.array([[0., 0.], [3., 4.], [6., 8.]])
    C_small = np.array([[0., 0.], [3., 0.]])
    got = np.asarray(KMeans(2).compute_distance(X_small, C_small), dtype=float)
    assert got.shape == (3, 2), f"expected shape (n, K) = (3, 2), got {got.shape}"
    assert np.allclose(got, [[0., 3.], [5., 4.], [10., 73 ** 0.5]]), f"wrong distances:\n{got}"
    print("compute_distance: local test passed")

test_distance()

In [ ]:
# PennGrader Grading Cell
grader.grade(test_case_id='test_distance', answer=grader_serialize(KMeans))

In [ ]:
# Local test: find_closest_cluster. Index of the smallest distance, as a flat (n,) array.
def test_find_clusters():
    dist_small = np.array([[1.0, 2.0, 3.0], [5.0, 0.5, 2.0], [4.0, 4.0, 0.1], [0.2, 9.0, 9.0]])
    got = np.asarray(KMeans(3).find_closest_cluster(dist_small))
    assert got.shape == (4,), f"expected a flat array of shape (4,), got {got.shape}"
    assert np.array_equal(got, [0, 1, 2, 0]), f"wrong cluster ids: {got}"
    print("find_closest_cluster: local test passed")

test_find_clusters()

In [ ]:
# PennGrader Grading Cell
grader.grade(test_case_id='test_find_clusters', answer=grader_serialize(KMeans))

In [ ]:
# Local test: fit. The hidden test checks the same four things on different data:
# the clusters and the centroids from a given start, an empty cluster, and init_centroids=None.
def test_fit():
    rng = np.random.RandomState(0)
    X_blobs = np.concatenate([rng.normal(c, 1.0, size=(30, 2))
                              for c in ([0.0, 0.0], [6.0, 6.0], [0.0, 8.0])])

    # 1 and 2. From a given start.
    clusters, centroids, iters_taken = KMeans(3).fit(X_blobs, np.array([[1., 1.], [5., 5.], [1., 7.]]))
    clusters, centroids = np.asarray(clusters), np.asarray(centroids, dtype=float)
    assert clusters.shape == (90,), f"clusters should have shape (90,), got {clusters.shape}"
    assert np.array_equal(clusters, np.repeat([0, 1, 2], 30)), "some points are in the wrong cluster"
    assert np.allclose(centroids, [[ 0.0801,  0.0733],
     [ 6.0562,  6.3209],
     [-0.1365,  8.1657]], atol=1e-3), f"wrong centroids:\n{centroids}"
    assert 1 <= int(iters_taken) <= 1000, f"iters_taken should be between 1 and max_iter, got {iters_taken}"

    # 3. The third centroid starts far from every point, so its cluster is empty.
    _, centroids, _ = KMeans(3).fit(X_blobs[:60], np.array([[1., 1.], [5., 5.], [50., 50.]]))
    centroids = np.asarray(centroids, dtype=float)
    assert not np.isnan(centroids).any(), "an empty cluster produced a NaN centroid"
    assert np.allclose(centroids, [[ 0.0801,  0.0733],
     [ 6.0562,  6.3209],
     [50.    , 50.    ]], atol=1e-3), \
        f"an empty cluster should keep its previous centroid:\n{centroids}"

    # 4. No start given: fit must use the provided init_centroids.
    _, centroids, _ = KMeans(3, random_state=3).fit(X_blobs)
    assert np.allclose(np.asarray(centroids, dtype=float), [[ 2.9599,  7.2433],
     [ 1.2058,  0.1153],
     [-0.6704,  0.0452]], atol=1e-3), \
        "with init_centroids=None, call self.init_centroids(X) and leave that method unchanged"
    print("fit: local test passed")

test_fit()

In [ ]:
# PennGrader Grading Cell
grader.grade(test_case_id='test_fit', answer=grader_serialize(KMeans))

## **1.3. The WCSS Curve [2 pts, Autograded]**

To choose $K$, a common approach is to run K-means for several values of $K$ and plot the
K-means objective, the WCSS, against $K$:

$$\text{WCSS} = \sum_{i=1}^{n} \lVert x_i - \mu_{f_\mu(x_i)} \rVert_2^2 ,$$

the sum, over all samples, of the **squared** Euclidean distance to the sample's own centroid. (You
may see this called *distortion* elsewhere.) WCSS always falls as $K$ grows, so we look for the
"knee" (or "elbow") where adding clusters stops helping much.

K-means only finds a local optimum, and which one depends on the initialization. As lecture
suggests, we therefore **run K-means several times from different initializations and keep the
best run**. Complete `wcss_curve`:

* for each $K = 2, 3, \dots,$ `max_k` (inclusive),
* run `KMeans(K, random_state=s).fit(X)` for each `s = 0, 1, ..., n_init - 1`,
* compute each run's WCSS, and keep the **lowest**.

Return a list of `max_k - 1` numbers: the best WCSS for $K = 2$, then $K = 3$, and so on.

That is 90 runs of K-means. With `compute_distance` written in numpy (no loop over samples) the
grading cell below takes a few seconds; if it takes minutes, vectorize `compute_distance` (Tip 3).
It runs again in 1.5, and again when Gradescope re-runs your notebook.

In [ ]:
def wcss_curve(X, max_k, n_init=10):
    """
    The lowest WCSS over n_init runs of K-means, for each K = 2, ..., max_k.

    INPUT:
    X - (n, d) array of samples
    max_k - the largest number of clusters to try
    n_init - number of runs per K, with random_state = 0, 1, ..., n_init - 1

    OUTPUT:
    scores - list of length max_k - 1; scores[K - 2] is the lowest WCSS found for K clusters
    """
    scores = []
    # STUDENT TODO START:
    raise NotImplementedError("1.3: wcss_curve")
    # STUDENT TODO END
    return scores

In [ ]:
# Local test: the same check as the hidden test -- every value of the curve -- on 90 points.
def test_wcss_curve():
    rng = np.random.RandomState(0)
    X_blobs = np.concatenate([rng.normal(c, 1.0, size=(30, 2))
                              for c in ([0.0, 0.0], [6.0, 6.0], [0.0, 8.0])])
    got = np.asarray(wcss_curve(X_blobs, max_k=4, n_init=3), dtype=float)
    assert got.shape == (3,), f"expected 3 values (K = 2, 3, 4), got {got.shape}"
    assert np.allclose(got, [1152.258,  179.114,  153.52 ], rtol=1e-4), f"wrong WCSS values: {got}"
    print("wcss_curve: local test passed")

test_wcss_curve()

In [ ]:
# PennGrader Grading Cell -- the raw features, K = 2..10, best of 10 runs each.
scores_raw = wcss_curve(cancer_dataset.data, max_k=10, n_init=10)
grader.grade(test_case_id='test_wcss_curve', answer=scores_raw)

## **1.4. Choose K for the Raw Features [2 pts, Manually Graded]**

Plot `scores_raw` from the grading cell above: WCSS on the y-axis against $K = 2, \dots, 10$ on
the x-axis. Give the plot a **title** and **axis labels**. (There is only one curve, so a legend
is optional.) Then, in the answer cell below it, say which $K$ you would choose and why, in one
or two sentences.

Marks: **2** — 1 for a correctly labelled plot of your `scores_raw`, 1 for a choice of $K$
justified from the plot. (The values themselves are graded once, in 1.3; a mistake there does not
cost this mark too.) Lecture calls choosing the knee "somewhat subjective": a reasonable
choice argued from the curve gets the mark.

In [ ]:
# Plot for 1.4 -- keep the figure in this cell; it is what gets marked.
# STUDENT TODO START:
raise NotImplementedError("1.4: plot scores_raw against K")
# STUDENT TODO END

**Your answer (1.4):**

*replace this italic line with your answer*

## **1.5. Choose K for the Standardized Features [2 pts, Manually Graded]**

So far K-means has run on the raw features. Now **standardize** each feature as in lecture
(L05): subtract its mean and divide by its standard deviation, so every feature has mean 0 and
standard deviation 1. Use
[`sklearn.preprocessing.StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
and store the result in **`X_std`** — Part 2 uses it too.

Then compute `scores_std = wcss_curve(X_std, max_k=10, n_init=10)` and plot it the same way
as in 1.4. In the answer cell, say which $K$ you would choose and why.

Marks: **2** — 1 for a correctly labelled plot of your `scores_std`, from the
standardized features, 1 for a justified choice of $K$. Here the knee is much less clear-cut than in 1.4; saying so, and why, is a good
answer.

In [ ]:
# Plot for 1.5 -- keep the figure in this cell; it is what gets marked.
from sklearn.preprocessing import StandardScaler

# STUDENT TODO START:
# 1. X_std = the standardized features (n, d)
# 2. scores_std = wcss_curve(X_std, max_k=10, n_init=10)
# 3. plot scores_std against K = 2, ..., 10, with a title and axis labels
raise NotImplementedError("1.5: standardize X, compute scores_std and plot it")
# STUDENT TODO END

**Your answer (1.5):**

*replace this italic line with your answer*

## **1.6. Feature Scaling and K-Means [5190 Extra — 3 pts, Manually Graded]**

Answer in the cell below, a few sentences each.

1. **[2 pts]** Why do the curves in 1.4 and 1.5 look so different? Look at the scale of the raw
   features (for instance `X.std(axis=0)`) and explain what that does to the Euclidean distance
   K-means uses.
2. **[1 pt]** Should you standardize features before running K-means? Give one situation where
   you would, and one where you might not.

**Your answer (1.6):**

*replace this italic line with your answer*

# **2. Principal Component Analysis [13 pts — 9 for CIS 4190]**

> **PCA is lectured on Tue 20 October, the day this homework is released.** Everything below
> uses the notation from that lecture.

PCA finds new axes for the data — unit vectors $v_1, v_2, \dots$, the **principal components** —
in order of how much of the data's variance each captures. With the data matrix $X$ ($n \times d$)
**mean-subtracted**, the components are the eigenvectors of the covariance matrix
$\Sigma = \frac{1}{n} X^\top X$, ordered by eigenvalue $\lambda_1 \ge \lambda_2 \ge \dots$, and a
sample is represented by its projections onto the first $d'$ of them:

$$f(x) = \left[x^\top v_1, \; x^\top v_2, \; \dots, \; x^\top v_{d'}\right] \in \mathbb{R}^{d'} .$$

Component $i$ explains a fraction $\lambda_i / \sum_j \lambda_j$ of the variance — its **explained
variance ratio**.

In this part you will use scikit-learn's
[`PCA`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html), which
mean-subtracts the data for you. **We standardize the features first, for the same reason as in
1.5:** PCA looks for directions of large variance, and on the raw features the direction of
largest variance is mostly a matter of which feature happens to be measured in the biggest units.

## **2.1. PCA as Preprocessing for Classification [5 pts — 2 Autograded, 3 Manually Graded]**

PCA is often used to shrink the input to a supervised model. Here you will predict the diagnosis
`y` with **logistic regression** — the classifier from lecture and HW1, here scikit-learn's
[`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
— first from all 30 features and then from the first $d'$ principal components, and see how
much accuracy the compression costs.

The split is provided below; do not change it. Then complete `evaluate_pca_logreg`:

1. Fit a `StandardScaler` on `X_train` **only**, and use it to transform both `X_train` and `X_test`.
2. **Baseline:** fit `LogisticRegression(max_iter=1000)` on the standardized training data and
   compute its accuracy on the standardized test data.
3. For each $d' = 1, 2, \dots,$ `max_components`: fit `PCA(n_components=d')` on the standardized
   **training** data only, project both splits with it, fit a new `LogisticRegression(max_iter=1000)`
   on the projected training data, and compute its accuracy on the projected test data.

Accuracy is the fraction of test samples predicted correctly. Fitting the scaler or the PCA on the
test data lets information from the test set into training, which is why both are fit on
`X_train` alone.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Provided -- do not change. 381 training and 188 test samples.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
print("train:", X_train.shape, " test:", X_test.shape)

In [ ]:
def evaluate_pca_logreg(X_train, X_test, y_train, y_test, max_components=10):
    """
    Test accuracy of logistic regression on all standardized features, and on the first
    d' principal components for d' = 1, ..., max_components.

    INPUT:
    X_train, X_test - raw (unstandardized) feature arrays from the provided split
    y_train, y_test - the matching labels
    max_components - the largest number of principal components to try

    OUTPUT:
    baseline_accuracy - float, test accuracy using all standardized features
    pca_accuracies - list of max_components floats; pca_accuracies[d - 1] is the test
                     accuracy using the first d principal components
    """
    # STUDENT TODO START:
    raise NotImplementedError("2.1: evaluate_pca_logreg")
    # STUDENT TODO END
    return baseline_accuracy, pca_accuracies

In [ ]:
baseline_accuracy, pca_accuracies = evaluate_pca_logreg(X_train, X_test, y_train, y_test,
                                                     max_components=10)
print("baseline:", round(float(baseline_accuracy), 4))
print("with 1..10 components:", [round(float(a), 4) for a in pca_accuracies])

In [ ]:
# Local test. These are sanity checks only: the hidden test compares every one of the 11
# numbers with the reference, allowing each to be off by one test sample (1/188). If one fails,
# you can still run the grading cell below and read the autograder's hint.
def test_pca_accuracies(baseline, accs):
    accs = np.asarray(accs, dtype=float)
    assert accs.shape == (10,), f"expected 10 accuracies, got {accs.shape}"
    assert 0.9 <= float(baseline) <= 1.0, f"a baseline of {baseline:.3f} is implausibly low"
    counts = np.append(accs, baseline) * len(y_test)
    assert np.allclose(counts, np.round(counts)), "accuracies should be fractions of the 188 test samples"
    assert accs[-1] >= 0.9, "with 10 components the accuracy should be close to the baseline"
    print("evaluate_pca_logreg: local sanity checks passed")

test_pca_accuracies(baseline_accuracy, pca_accuracies)

In [ ]:
# PennGrader Grading Cell
grader.grade(test_case_id='test_pca_accuracies',
             answer=[float(baseline_accuracy)] + [float(a) for a in pca_accuracies])

### Plot the accuracies [3 pts, Manually Graded]

Plot `pca_accuracies` against the number of components $d' = 1, \dots, 10$, and add a **black,
dashed horizontal line** at `baseline_accuracy`. Give the plot a title, axis labels and a legend.
Then, in the answer cell, describe the trend, explain it in terms of what the first few
components capture, and say what $d'$ you would use and why.

Marks: **3** — 1 for the plot, 2 for the comment (the trend and its explanation, and a
justified choice of $d'$).

In [ ]:
# Plot for 2.1 -- keep the figure in this cell; it is what gets marked.
# STUDENT TODO START:
raise NotImplementedError("2.1: plot pca_accuracies with the baseline as a dashed line")
# STUDENT TODO END

**Your answer (2.1):**

*replace this italic line with your answer*

## **2.2. Understanding PCA [4 pts]**

### **2.2.1. How Many Components? [2 pts, Autograded]**

Complete `select_n_principal_components`. Given data and a threshold `variation` between 0 and 1,
return the **smallest number of components** $d'$ whose explained variance ratios add up to at
least `variation`:

$$\min \left\{ d' \;:\; \textstyle\sum_{i=1}^{d'} \lambda_i / \sum_j \lambda_j \ \ge\ \text{variation} \right\} .$$

Fit `PCA()` with all components and use its `explained_variance_ratio_`. Return a count — a
plain `int` — not an index. The grading cell runs your function on `X_std` from 1.5.

In [ ]:
def select_n_principal_components(data, variation):
    """
    The smallest number of principal components that explains at least `variation`
    of the variance of `data`.

    INPUT:
    data - (n, d) array (already standardized, if you want it standardized)
    variation - float strictly between 0 and 1

    OUTPUT:
    n_components - int, between 1 and d
    """
    # STUDENT TODO START:
    raise NotImplementedError("2.2.1: select_n_principal_components")
    # STUDENT TODO END
    return n_components

In [ ]:
# Local test: the same check as the hidden test, on data with a known spread of variances.
def test_select_n_principal_components():
    rng = np.random.RandomState(1)
    data = rng.normal(size=(500, 4)) * np.array([3.0, 2.0, 1.0, 0.5])
    got = [select_n_principal_components(data, v) for v in (0.6, 0.9, 0.99)]
    assert got == [1, 2, 4], f"expected [1, 2, 4], got {got}"
    print("select_n_principal_components: local test passed")

test_select_n_principal_components()

# For comparison (not graded): the same question on the RAW features.
print("raw features, 80/90/95/99%:", [select_n_principal_components(X, v) for v in (0.80, 0.90, 0.95, 0.99)])

In [ ]:
# PennGrader Grading Cell -- the standardized features from 1.5.
grader.grade(test_case_id='test_select_n_principal_components',
             answer=[select_n_principal_components(X_std, v) for v in (0.80, 0.90, 0.95, 0.99)])

### **2.2.2. What Do the First Two Components Measure? [2 pts, Manually Graded]**

Each principal component is a weighted combination of the original features; its weights (the
*loadings*) are the entries of $v_j$, one per feature, and sklearn stores them as the rows of
`pca.components_`. Features with large loadings, positive or negative, are the ones the component
mostly measures.

Fit a 2-component PCA on `X_std` below. The rest of the cell, provided, lists the three features
with the largest absolute loading on each component. In the answer cell, for **PC1** and for
**PC2**: name those three features, and say in one sentence what kind of variation between tumours
that component captures.

Marks: **2** — 1 per component.

In [ ]:
# Loadings for 2.2.2
# STUDENT TODO START:
# pca_2d = a PCA with 2 components, fit on X_std
raise NotImplementedError("2.2.2: fit a 2-component PCA on X_std")
# STUDENT TODO END

loadings = pd.DataFrame(pca_2d.components_.T, index=feature_names, columns=["PC1", "PC2"])
for pc in ["PC1", "PC2"]:
    top = loadings[pc].abs().sort_values(ascending=False).index[:3]
    print(f"{pc} explains {pca_2d.explained_variance_ratio_[int(pc[-1]) - 1]:.1%} of the variance.")
    print(f"  largest loadings: " + ", ".join(f"{name} ({loadings.loc[name, pc]:+.3f})" for name in top))

**Your answer (2.2.2):**

*replace this italic line with your answer*

## **2.3. K-Means on the PCA Plane [5190 Extra — 1 pt Autograded, 3 pts Manually Graded]**

PCA and K-means are often used together: PCA to look at high-dimensional data, K-means to find
groups in it. Here you will run **your** K-means on all 30 standardized features, then look at
the clusters in the plane of the first two principal components.

Complete the two functions below. `kmeans_clusters(X, K)` returns your K-means cluster ids for
`X` (use `KMeans(K, random_state=0)`). `agreement_with_labels(clusters, y)` returns, for $K = 2$, the
fraction of samples whose cluster matches the diagnosis. Cluster ids are arbitrary — cluster 0
might be the benign one — so take the better of the two ways of pairing clusters with labels.

The plotting cell after them is provided. It runs your K-means on `X_std` for $K$ in `[2, 3, 5]` and
draws each clustering in the PC1–PC2 plane, next to the true diagnosis. Because 2.3 is
[5190 Extra], that cell prints a note and moves on until both functions are written, so leaving
2.3 out never stops the rest of the notebook from running.

Answer in the cell below:

1. **[1 pt, autograded]** `agreement` itself — the grading cell after the plot submits it.
   **[1 pt]** Comment on it: did K-means, which never saw `y`, find the diagnosis? Explain why you
   take the better of the two matchings.
2. **[2 pts]** In some panels, points of different colours overlap in the PC1–PC2 plane. How can
   K-means separate points that sit on top of each other in this plot? (Hint: how much of the
   variance do two components keep, according to 2.2.2?)

In [ ]:
def kmeans_clusters(X, K):
    """
    Your K-means from 1.2, run on X with K clusters and random_state=0.

    OUTPUT:
    clusters - (n,) integer array of cluster ids
    """
    # STUDENT TODO START:
    raise NotImplementedError("2.3: kmeans_clusters")
    # STUDENT TODO END
    return clusters


def agreement_with_labels(clusters, y):
    """
    For K = 2: the fraction of samples whose cluster id matches their label in y, under the
    better of the two ways of pairing the cluster ids {0, 1} with the labels {0, 1}.

    OUTPUT:
    agreement - float between 0.5 and 1
    """
    # STUDENT TODO START:
    raise NotImplementedError("2.3: agreement_with_labels")
    # STUDENT TODO END
    return agreement

In [ ]:
# Plot for 2.3 -- keep the figure in this cell; it is what gets marked.
# Provided. Until both functions above are written, this prints a note instead of stopping.
try:
    panels = {K: np.asarray(kmeans_clusters(X_std, K)) for K in [2, 3, 5]}
    agreement = agreement_with_labels(panels[2], y)
except NotImplementedError as e:
    print(f"2.3 skipped -- {e} is not written yet. (2.3 is [5190 Extra]: optional for CIS 4190.)")
else:
    X_pca_2d = PCA(n_components=2).fit_transform(X_std)
    fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
    for label, name, colour in [(0, "malignant", "tab:red"), (1, "benign", "tab:blue")]:
        axes[0].scatter(X_pca_2d[y == label, 0], X_pca_2d[y == label, 1], c=colour, s=12, label=name)
    axes[0].legend()
    axes[0].set_title("True diagnosis")
    for ax, K in zip(axes[1:], [2, 3, 5]):
        ax.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=panels[K], cmap="viridis", s=12)
        ax.set_title(f"K-means clusters, K = {K}")
    for ax in axes:
        ax.set_xlabel("PC1")
        ax.set_ylabel("PC2")
    plt.tight_layout()
    plt.show()
    print(f"K = 2 agreement with the diagnosis: {agreement:.3f}")

In [ ]:
# Local test for 2.3: the better-of-two-pairings rule on a five-sample example, and the shape of
# your clusters. (The hidden test checks the agreement on the real data.) Prints a note instead
# until 2.3's functions are written.
def test_kmeans_agreement():
    got = agreement_with_labels(np.array([0, 0, 1, 1, 1]), np.array([1, 1, 0, 0, 1]))
    assert abs(float(got) - 0.8) < 1e-9, \
        f"clusters [0,0,1,1,1] against labels [1,1,0,0,1] should give 0.8 (the better pairing), got {got}"
    clusters = np.asarray(kmeans_clusters(X_std, 2))
    assert clusters.shape == (X_std.shape[0],), f"kmeans_clusters should return shape (569,), got {clusters.shape}"
    print("2.3: local test passed")

try:
    test_kmeans_agreement()
except NotImplementedError as e:
    print(f"2.3 skipped -- {e} is not written yet. (2.3 is [5190 Extra]: optional for CIS 4190.)")

In [ ]:
# PennGrader Grading Cell -- 2.3 [5190 Extra]. Prints a note instead until 2.3's functions exist.
try:
    _agreement = float(agreement_with_labels(kmeans_clusters(X_std, 2), y))
except NotImplementedError as e:
    print(f"2.3 skipped -- {e} is not written yet. (2.3 is [5190 Extra]: optional for CIS 4190.)")
else:
    grader.grade(test_case_id='test_kmeans_agreement', answer=_agreement)

**Your answer (2.3):**

*replace this italic line with your answer*

# **3. Image Classification with a CNN [14 pts]**

> **This part is lectured during the homework window.** Convolutional networks are covered on
> **Thu 22 October** (CNNs Part 1) and **Tue 27 October** (CNNs Part 2), after this homework is
> released. Sections 3.1 and 3.2 — loading the images — need nothing from those lectures, so you
> can start them now.

You will train a convolutional neural network to classify images from the racing game
[SuperTuxKart](https://supertuxkart.net/) into **6 classes**: `background` (0), `kart` (1),
`pickup` (2), `nitro` (3), `bomb` (4) and `projectile` (5). Every image is $64 \times 64$ RGB.
There are 21,000 training, 6,000 validation and 3,000 test images; the test labels are held back,
and the autograder scores your predictions on them.

**A GPU runtime makes this part much faster** (*Runtime → Change runtime type → T4 GPU*); a CPU
works too, more slowly. If you switch, re-run every cell from the top of the notebook down to
*Initialize the Autograder* — Part 3 needs nothing else from Parts 1 and 2.

**About `if NOTEBOOK:`.** Gradescope re-runs your notebook on a machine without a GPU or the image
data. So the cells in this part that download data, train or predict are wrapped in
`if NOTEBOOK:`, and Gradescope skips them. Write your code in the functions and classes provided —
they are not guarded — and leave the guards as they are. **If you add cells of your own that read
the images or train, put them under `if NOTEBOOK:` too**: on Gradescope there are no images, and an
error stops the re-run before the report cells at the end. It also means **Gradescope cannot record
3.5's grading cell for you: run it in Colab.** (3.1's grading cell is not guarded — it needs no
images — so Gradescope re-runs that one like those in Parts 1 and 2.)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

torch.manual_seed(42)  # don't change this line!!

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

#### Download and extract the data

In [ ]:
import zipfile

DATA_ZIP = "cis5190_hw3_supertuxkart_data.zip"
DATA_URL = "https://raw.githubusercontent.com/CIS5190/cis-4190-5190-fall-26/main/hw3/" + DATA_ZIP

if NOTEBOOK:
    if not os.path.exists(DATA_ZIP):
        !wget -q {DATA_URL}
    if not os.path.exists("train"):
        with zipfile.ZipFile(DATA_ZIP) as zf:
            zf.extractall(".")
    print({split: len(os.listdir(split)) - 1 for split in ["train", "val", "test"]}, "images")

## **3.1. The Dataset Class [2 pts, Autograded]**

PyTorch reads data through a [`Dataset`](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html).
Each of the `train/`, `val/` and `test/` folders holds the images plus a `labels.csv` with one row
per image. In `train/labels.csv` and `val/labels.csv` the columns are `file`, `label` (a class
*name* such as `kart`) and `track`; `test/labels.csv` has no `label` column. Open one with `pandas`
to see it before you start.

Complete the `STKDataset` class:

1. `__len__` returns the number of images in the folder.
2. `__getitem__(idx)` builds the path to image `idx` by joining `self.image_path` with the
   file name in the `file` column (use `os.path.join`), and — for a folder with labels — encodes the
   label name as an `int` with `LABELS_TO_ENCODING`. It returns `(image, label)`, or the image
   alone for the test folder. The rest of the method is provided.

Then define `transform` so that each image becomes a `torch.Tensor` of shape `(3, 64, 64)` with
every pixel value in $[-1, 1]$. Use
[`transforms.Compose`](https://pytorch.org/vision/stable/generated/torchvision.transforms.Compose.html)
with `transforms.ToTensor()`, which gives values in $[0, 1]$, followed by `transforms.Normalize`.

Marks: **2** — 1 for the dataset class, 1 for the transform. They are checked on a
tiny folder of plain-coloured images that the probe cell below creates, so you can submit this
section before you have downloaded the real data.

In [ ]:
ENCODING_TO_LABELS = {0: "background",
                      1: "kart",
                      2: "pickup",
                      3: "nitro",
                      4: "bomb",
                      5: "projectile"}

LABELS_TO_ENCODING = {name: code for code, name in ENCODING_TO_LABELS.items()}

In [ ]:
class STKDataset(Dataset):

    def __init__(self, image_path, transform=None):
        self.image_path = image_path
        self.labels = pd.read_csv(os.path.join(image_path, "labels.csv"))
        self.transform = transform

    def __len__(self):
        # STUDENT TODO START:
        raise NotImplementedError("3.1: __len__")
        # STUDENT TODO END

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        # STUDENT TODO START:
        # img_name = the full path of image idx
        raise NotImplementedError("3.1: build img_name")
        # STUDENT TODO END

        image = Image.open(img_name)
        if self.transform:
            image = self.transform(image)

        if "label" in self.labels.columns:
            # STUDENT TODO START:
            # label = the int code of image idx's label name
            raise NotImplementedError("3.1: encode the label")
            # STUDENT TODO END
            return image, label
        return image

In [ ]:
# STUDENT TODO START:
# transform = a transforms.Compose that gives (3, 64, 64) tensors with values in [-1, 1]
raise NotImplementedError("3.1: define transform")
# STUDENT TODO END

In [ ]:
# For autograder only, do not modify this cell.
# Builds a three-image folder of plain colours and records what your STKDataset returns for it
# and what your transform makes of each image -- separately, so each is marked on its own.
# Runs anywhere: it does not need the downloaded data.
import tempfile

def _stk_probe():
    out = {}

    def record(key, get):
        try:
            out[key] = get()
        except NotImplementedError as e:
            out[key] = f"not implemented yet ({e})"
        except Exception as e:
            out[key] = f"{type(e).__name__}: {e}"

    root = tempfile.mkdtemp()
    colours = [(0, 0, 0), (255, 255, 255), (255, 0, 128)]
    names = ["a.png", "b.png", "c.png"]
    for split in ["probe_train", "probe_test"]:
        os.makedirs(os.path.join(root, split))
        for name, colour in zip(names, colours):
            Image.new("RGB", (64, 64), colour).save(os.path.join(root, split, name))
    pd.DataFrame({"file": names, "label": ["nitro", "background", "projectile"],
                  "track": ["probe"] * 3}).to_csv(os.path.join(root, "probe_train", "labels.csv"), index=False)
    pd.DataFrame({"file": names, "track": ["probe"] * 3}).to_csv(
        os.path.join(root, "probe_test", "labels.csv"), index=False)

    train_dir, test_dir = os.path.join(root, "probe_train"), os.path.join(root, "probe_test")

    # The dataset class, checked with no transform -- so a mistake in `transform` cannot
    # cost the dataset point, and the other way round.
    def item(i):
        img, label = STKDataset(train_dir)[i]
        return {"label": label if isinstance(label, (int, str)) else int(label),
                "label_type": type(label).__name__}

    def test_item():
        return {"is_tuple": isinstance(STKDataset(test_dir)[1], tuple)}

    # The transform, checked on its own, on the same three images.
    def pixels(i):
        t = torch.as_tensor(transform(Image.open(os.path.join(train_dir, names[i]))),
                            dtype=torch.float32)
        return {"shape": list(t.shape), "min": float(t.min()), "max": float(t.max()),
                "mean": float(t.mean())}

    record("len", lambda: int(len(STKDataset(train_dir))))
    for i in range(3):
        record(f"item{i}", lambda i=i: item(i))
        record(f"pixels{i}", lambda i=i: pixels(i))
    record("test_item", test_item)
    return out

STK_PROBE = _stk_probe()
for key, value in STK_PROBE.items():
    print(f"{key:>9}: {value}")

In [ ]:
# Local test: the same checks as the hidden test, on the probe's three images.
def test_stk_dataset():
    p = STK_PROBE
    unfinished = [v for v in p.values() if isinstance(v, str)
                  and ("not implemented" in v or "is not defined" in v)]
    if unfinished:
        raise NotImplementedError(f"3.1 is not finished yet -- the probe reports: {unfinished[0]}")
    assert p["len"] == 3, f"__len__ should be 3 for the probe folder, got {p['len']}"
    for i, label in enumerate([3, 0, 5]):        # nitro, background, projectile
        item = p[f"item{i}"]
        assert isinstance(item, dict), f"STKDataset __getitem__({i}) failed: {item}"
        assert item["label_type"].startswith("int"), f"labels should be ints, got {item['label_type']}"
        assert item["label"] == label, f"item {i}: expected label {label}, got {item['label']}"
    assert isinstance(p["test_item"], dict) and not p["test_item"]["is_tuple"], \
        "an item from a folder with no 'label' column should be the image alone"
    # The transform: black -> -1, white -> +1, and a coloured image keeps its colour channels.
    for i, (lo, hi) in enumerate([(-1.0, -1.0), (1.0, 1.0)]):
        px = p[f"pixels{i}"]
        assert isinstance(px, dict), f"transform failed on image {i}: {px}"
        assert px["shape"] == [3, 64, 64], f"image shape should be [3, 64, 64], got {px['shape']}"
        assert abs(px["min"] - lo) < 0.01 and abs(px["max"] - hi) < 0.01, \
            f"image {i}: pixel values should map to [-1, 1] (black -1, white +1), got {px['min']}..{px['max']}"
    px = p["pixels2"]
    assert isinstance(px, dict) and abs(px["mean"] - 0.0013) < 0.01, \
        f"a coloured image (255, 0, 128) should keep its three channels after the transform: {px}"
    print("STKDataset and transform: local test passed")

test_stk_dataset()

In [ ]:
# PennGrader Grading Cell
grader.grade(test_case_id='test_stk_dataset', answer=STK_PROBE)

In [ ]:
if NOTEBOOK:
    train_dataset = STKDataset(image_path="train", transform=transform)
    val_dataset = STKDataset(image_path="val", transform=transform)
    test_dataset = STKDataset(image_path="test", transform=transform)

    # Visual sanity check: nine training images with their labels.
    figure = plt.figure(figsize=(8, 8))
    g = torch.Generator().manual_seed(0)
    for i in range(1, 10):
        sample_idx = torch.randint(len(train_dataset), size=(1,), generator=g).item()
        img, label = train_dataset[sample_idx]
        figure.add_subplot(3, 3, i)
        plt.title(ENCODING_TO_LABELS[label])
        plt.axis("off")
        plt.imshow(img.permute(1, 2, 0) * 0.5 + 0.5)   # undo the normalization for display
    plt.show()

## **3.2. Data Loaders [ungraded]**

Complete `make_dataloaders` so it returns a [`DataLoader`](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader)
for each split, with batch size `batch_size`. **Shuffle the training loader only.** The test loader
must keep `test/labels.csv`'s order, because the autograder matches your predictions to the held-back
labels position by position: a shuffled test loader scores about one in six.

In [ ]:
def make_dataloaders(train_dataset, val_dataset, test_dataset, batch_size=64):
    """
    OUTPUT:
    train_dataloader - shuffled
    val_dataloader, test_dataloader - not shuffled
    """
    # STUDENT TODO START:
    raise NotImplementedError("3.2: make_dataloaders")
    # STUDENT TODO END
    return train_dataloader, val_dataloader, test_dataloader

In [ ]:
if NOTEBOOK:
    train_dataloader, val_dataloader, test_dataloader = make_dataloaders(
        train_dataset, val_dataset, test_dataset, batch_size=64)
    print(len(train_dataloader), "training batches")

## **3.3. The CNN Architecture**

Design a CNN in the class `Net`. Its input is a batch of shape `(batch_size, 3, 64, 64)` and its
output must have shape `(batch_size, 6)`: one score (logit) per class. Do **not** apply a softmax at
the end — `nn.CrossEntropyLoss` does that for you.

The choices are yours: the number of convolutional layers; each one's kernel size, stride, padding
and number of output channels; pooling; and the fully connected layers after them. To keep the
search manageable we suggest **at most four convolutional and four fully connected layers**, with a
**ReLU** between layers. Lecture's rule for the spatial size after a layer is a good way to work out
the input size of your first fully connected layer.

Section 3.5 grades the network's accuracy on the test images, so the architecture is worth some
experimenting — with the validation accuracy, not the test set, as your guide.

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # STUDENT TODO START:
        # Create the layers of your CNN here
        raise NotImplementedError("3.3: define the layers")
        # STUDENT TODO END

    def forward(self, x):
        # STUDENT TODO START:
        # x has shape (batch_size, 3, 64, 64); return logits of shape (batch_size, 6)
        raise NotImplementedError("3.3: forward pass")
        # STUDENT TODO END

In [ ]:
torch.manual_seed(42)  # don't change this line!! (re-running this cell restarts from the same weights)
model = Net().to(device)
# If you re-run this cell, re-run 3.4's criterion/optimizer cell too: an optimizer built for the
# previous model would keep updating that one, and this model would never train.

# Check the output shape on a dummy batch of two images. (eval mode, so a BatchNorm layer, if
# you use one, does not fold this fake batch into its running statistics.)
model.eval()
with torch.no_grad():
    out = model(torch.zeros(2, 3, 64, 64, device=device))
model.train()
assert tuple(out.shape) == (2, 6), f"Net should output shape (batch_size, 6), got {tuple(out.shape)}"
print("Net output shape OK;", sum(p.numel() for p in model.parameters()), "parameters")

## **3.4. Training and Validation**

1. Set `criterion` to the cross-entropy loss and choose an `optimizer` for `model.parameters()`.
   `optim.Adam` with its default learning rate is a good start; try others if you like.
2. Complete `train_one_epoch`: one pass over the training loader that, for every batch, moves the
   inputs and labels to `device`, computes the model's predictions and the loss, zeroes the
   gradients, back-propagates, and takes an optimizer step. Return the average loss over batches
   and the accuracy over all training samples.
3. Complete `evaluate`: the same pass over a loader **without** updating the model (use
   `model.eval()` and `torch.no_grad()`), returning the average loss and the accuracy.

The provided loop then trains for `num_epochs` epochs and plots the curves. Ten epochs is a
reasonable default; change `num_epochs` if your validation accuracy says you should.

In [ ]:
# STUDENT TODO START:
# criterion = ...
# optimizer = ...
raise NotImplementedError("3.4: criterion and optimizer")
# STUDENT TODO END

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    """
    One pass over dataloader, updating the model.

    OUTPUT:
    avg_loss - float, the mean of the batch losses
    accuracy - float, the fraction of samples predicted correctly
    """
    model.train()
    # STUDENT TODO START:
    raise NotImplementedError("3.4: train_one_epoch")
    # STUDENT TODO END
    return avg_loss, accuracy


def evaluate(model, dataloader, criterion, device):
    """
    One pass over dataloader WITHOUT updating the model.

    OUTPUT:
    avg_loss - float, the mean of the batch losses
    accuracy - float, the fraction of samples predicted correctly
    """
    model.eval()
    # STUDENT TODO START:
    raise NotImplementedError("3.4: evaluate")
    # STUDENT TODO END
    return avg_loss, accuracy

In [ ]:
num_epochs = 10

if NOTEBOOK:
    _model_params = {id(p) for p in model.parameters()}
    assert any(id(p) in _model_params for g in optimizer.param_groups for p in g["params"]), \
        "optimizer belongs to an older model -- re-run 3.4's criterion/optimizer cell after re-creating the model"

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(model, train_dataloader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_dataloader, criterion, device)
        for key, value in zip(history, [train_loss, val_loss, train_acc, val_acc]):
            history[key].append(value)
        print(f"Epoch {epoch + 1:>2}: train loss {train_loss:.3f}, acc {train_acc:.3f} | "
              f"val loss {val_loss:.3f}, acc {val_acc:.3f}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    epochs = range(1, num_epochs + 1)
    ax1.plot(epochs, history["train_loss"], label="train")
    ax1.plot(epochs, history["val_loss"], label="validation")
    ax1.set_xlabel("epoch"); ax1.set_ylabel("loss"); ax1.set_title("Loss"); ax1.legend()
    ax2.plot(epochs, history["train_acc"], label="train")
    ax2.plot(epochs, history["val_acc"], label="validation")
    ax2.set_xlabel("epoch"); ax2.set_ylabel("accuracy"); ax2.set_title("Accuracy"); ax2.legend()
    plt.show()

## **3.5. Predict the Test Set [12 pts, Autograded]**

Complete `predict`: run the model over the test loader (which yields images only, no labels) and
return the predicted class of every test image as a flat integer `numpy` array of shape `(3000,)`,
in the loader's order. Use `model.eval()` and `torch.no_grad()`, and take the class with the
highest score.

**Scoring.** You get the full **12 points** for a test accuracy of
**85% or more** and 0 for 60% or less, linearly in between, rounded to the
nearest whole point — so 84.0% already rounds to the full 12.
**This section asks for a convolutional network: if your `Net` has no `nn.Conv2d` layer, it is
capped at 6 points**, whatever its accuracy. For reference: always guessing one
class scores about 17%, a network with no convolutional layers reaches roughly 75–80%, and a small
CNN with pooling clears 85% within a few epochs. Your validation accuracy is a good estimate of your test accuracy, and it
moves by a few points from one epoch to the next — so look at the whole curve, not just the
last epoch, when you choose `num_epochs`.

In [ ]:
def predict(model, dataloader, device):
    """
    OUTPUT:
    predictions - (N,) integer numpy array: the predicted class of each image, in order
    """
    model.eval()
    # STUDENT TODO START:
    raise NotImplementedError("3.5: predict")
    # STUDENT TODO END
    return predictions

In [ ]:
if NOTEBOOK:
    test_predictions = predict(model, test_dataloader, device)

    # Local sanity checks. The hidden test compares these with the held-back labels.
    test_predictions = np.asarray(test_predictions)
    assert test_predictions.shape == (3000,), f"expected shape (3000,), got {test_predictions.shape}"
    assert set(np.unique(test_predictions)) <= set(range(6)), "predictions should be class ids 0-5"
    print("Predicted class counts:", np.bincount(test_predictions.astype(int), minlength=6))

    conv_layers = sum(isinstance(m, nn.Conv2d) for m in model.modules())
    print(f"Your Net has {conv_layers} convolutional layer(s).")
    if conv_layers == 0:
        print("  -> with no nn.Conv2d layer, 3.5 is capped at 6 of 12 points.")

In [ ]:
# PennGrader Grading Cell -- run this in Colab; Gradescope skips it.
if NOTEBOOK:
    grader.grade(test_case_id='test_cnn_predictions',
                 answer={"predictions": test_predictions,
                         "conv_layers": sum(isinstance(m, nn.Conv2d) for m in model.modules())})

# **4. Optional: An LLM as the Classifier [Ungraded]**

> **This section is not graded, and it only runs on your own computer.** The course LLM can only
> be reached from PennNet, so run this section in Jupyter on your own computer while you are on
> campus or connected to the Penn VPN. It does not work in Colab, whose servers are outside
> PennNet. **A similar exercise may be graded in a later homework**, so it is worth trying now.

In 2.1 you *trained* logistic regression on the first two principal components. Here a large
language model (LLM) does the same job without any training. The provided prompt shows it some
labelled training points, and it labels 40 test points from the prompt alone. This is called
**in-context learning**. The prompt is written for you; what you control is which examples the
LLM sees.

**Running it locally.** Download this notebook (*File → Download → Download .ipynb*) and open it
in Jupyter on your own computer, with numpy and scikit-learn installed
(`pip install numpy scikit-learn notebook`). The cells in this section do not depend on anything
above them, so you only need to run these cells.

**Your LLM key.** Log in at
[https://coldfront.parcc.upenn.edu/user/login?next=/litellm-keys/](https://coldfront.parcc.upenn.edu/user/login?next=/litellm-keys/)
to get your key, then put it in place of `xxxx` in `key = "xxxx"` in the first cell below.
**Each person's key has a limited quota. Use it only for this homework.**

1. In the first cell below, `example_counts` lists how many malignant and how many benign
   training points go into the prompt, one LLM call per pair. The first pair, `(20, 20)`, is
   balanced. Add at least two pairs in which one class has far more examples than the other,
   such as `(35, 5)`. The training set has 145 malignant and 236 benign tumours; use at most 60
   of each.
2. Run the provided cell. For each pair it prints the LLM's accuracy, how many of the malignant
   test tumours it found, and how many tumours it called malignant, next to logistic regression
   trained on exactly the same examples.

The LLM's answers can change from run to run.

In [ ]:
# Your key and your choice of examples for section 4.
key = "xxxx"    # replace xxxx with your own key -- limited quota, for this homework only

# Each pair is (malignant examples, benign examples) put into the prompt for one LLM call.
# The first pair is balanced. Add at least two pairs in which one class has far more
# examples than the other, for example with example_counts.append((35, 5)).
example_counts = [(20, 20)]
# STUDENT TODO START:
# example_counts.append((..., ...))
# STUDENT TODO END

In [ ]:
# LLM results -- provided; run it after filling in the cell above, on your own computer on PennNet.
# Self-contained: it reloads the data and redoes 2.1's split, standardization and PCA.
import json
import os
import re
import sys
import urllib.error
import urllib.request

import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

LLM_URL = "https://litellm.parcc.upenn.edu/v1/chat/completions"


def ask_llm(prompt):
    """Sends one prompt to the course LLM with your key and returns its answer as text."""
    request = urllib.request.Request(
        LLM_URL, headers={"Authorization": "Bearer " + str(key).strip(), "Content-Type": "application/json"},
        data=json.dumps({"model": "openai/gpt-oss-120b", "temperature": 0, "reasoning_effort": "low",
                         "messages": [{"role": "user", "content": prompt}]}).encode())
    with urllib.request.urlopen(request, timeout=120) as response:
        return json.load(response)["choices"][0]["message"]["content"]


def llm_classifier_table(pairs):
    """One LLM call per (malignant, benign) pair; prints the first prompt and a table of results."""
    try:                                                         # fail fast when not on PennNet
        urllib.request.urlopen(LLM_URL.rsplit("/chat/", 1)[0] + "/models", timeout=5)
    except urllib.error.HTTPError:
        pass                                                     # the server answered: reachable
    except OSError as e:
        raise RuntimeError(f"the course LLM did not answer within 5 seconds ({e})") from None
    X_all, y_all = load_breast_cancer(return_X_y=True)                  # 2.1's data and split
    X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.33, random_state=42)
    scaler = StandardScaler().fit(X_tr)
    pca = PCA(n_components=2).fit(scaler.transform(X_tr))
    Z_train, Z_test = pca.transform(scaler.transform(X_tr)), pca.transform(scaler.transform(X_te))
    names = np.array(["malignant", "benign"])
    queries = np.random.default_rng(1).choice(len(y_te), 40, replace=False)  # the same 40 each time
    truth = y_te[queries]
    pools = [np.random.default_rng(0).permutation(np.where(y_tr == c)[0]) for c in (0, 1)]

    def describe(p):
        if p is None:
            return f"{'-':>9}{'-':>12}{'-':>9}"
        found = f"{np.sum(p[truth == 0] == 0)} of {np.sum(truth == 0)}"
        return f"{np.mean(p == truth):9.2f}{found:>12}{np.sum(p == 0):9d}"

    rows = []
    for n_mal, n_ben in pairs:
        shots = np.concatenate([pools[0][:n_mal], pools[1][:n_ben]])
        shots = shots[np.random.default_rng(2).permutation(len(shots))]
        prompt = ("Each point is a breast tumour, given by its first two principal components (PC1, PC2). "
                  "Classify each query point as malignant or benign.")
        if len(shots):
            prompt += "\n\nLabelled examples:\n" + "\n".join(
                f"({a:.2f}, {b:.2f}) -> {names[t]}" for (a, b), t in zip(Z_train[shots], y_tr[shots]))
        prompt += ("\n\nQuery points:\n"
                   + "\n".join(f"Q{i + 1}: ({a:.2f}, {b:.2f})" for i, (a, b) in enumerate(Z_test[queries]))
                   + "\n\nAnswer with exactly 40 lines, one per query, in the form "
                     "'Q<number>: malignant' or 'Q<number>: benign'. No other text.")
        if not rows:
            print("The prompt for your first pair:\n" + "-" * 72 + "\n" + prompt + "\n" + "-" * 72 + "\n")
        preds = np.full(len(queries), -1)                        # -1 = no answer, counted as wrong
        for m in re.finditer(r"Q(\d+)\s*[:.\-]\s*\**\s*(malignant|benign)", ask_llm(prompt), re.I):
            if 1 <= int(m.group(1)) <= len(queries):
                preds[int(m.group(1)) - 1] = 0 if m.group(2).lower() == "malignant" else 1
        lr = None                                                # logistic regression needs both classes
        if n_mal and n_ben:
            lr = LogisticRegression(max_iter=1000).fit(Z_train[shots], y_tr[shots]).predict(Z_test[queries])
        rows.append(f"  {f'{n_mal} + {n_ben}':<11}{describe(preds)}   {describe(lr)}")

    print(f"{'':13}{'LLM':-^30}   {'logistic regression':-^30}")
    print(f"  {'examples':<11}" + "   ".join([f"{'accuracy':>9}{'malignant':>12}{'said':>9}"] * 2))
    print(f"  {'mal + ben':<11}" + "   ".join([f"{'':>9}{'found':>12}{'malig.':>9}"] * 2))
    print("\n".join(rows))
    print(f"\nThe 40 test points include {np.sum(truth == 0)} malignant tumours. "
          f"Always answering 'benign' would score {np.mean(truth == 1):.2f}.")


# Nothing below may raise: an error here would stop Gradescope's re-run of the notebook.
try:
    pairs = [(int(m), int(b)) for m, b in globals().get("example_counts", [])]
except (TypeError, ValueError):
    pairs = []
if os.getenv("IS_AUTOGRADER") is not None:
    print("Section 4 skipped -- it is ungraded, and Gradescope does not run it.")
elif "google.colab" in sys.modules:
    print("Section 4 does not run in Colab: the course LLM can only be reached from PennNet.\n"
          "Run this section in Jupyter on your own computer, on campus or on the Penn VPN.")
elif str(globals().get("key", "")).strip() in ("", "xxxx"):
    print("Section 4 not run yet -- replace xxxx with your own key in the cell above.")
elif len(pairs) < 3 or not all(0 <= m <= 60 and 0 <= b <= 60 for m, b in pairs):
    print("Section 4 not run yet -- example_counts needs at least three (malignant, benign) pairs, "
          "each number between 0 and 60.")
else:
    try:
        llm_classifier_table(pairs)
    except Exception as e:                                       # noqa: BLE001
        print(f"Section 4 could not finish: {e}\nCheck your key, and that you are on PennNet (on campus "
              "or on the Penn VPN), then run this cell again.")

### Something to think about (not graded)

When one class has far more examples than the other, how do the LLM's answers change? Which of
these mistakes would be worse for a real patient?

**Your notes (section 4, not graded):**

*you can write your thoughts here*

# **Before You Submit: Two Short Reports [4 pts, Autograded]**

These two cells are worth **2 points each**, and the points are for *answering*,
not for any particular answer. There is no answer that scores better than
another: reporting that you used no AI earns full marks, and so does reporting
that you used a lot of it.

We use the responses to calibrate how long assignments actually take and to
understand how students are using AI tools.

### **1. Use of AI tools [2 pts]**

Report any use of AI tools (ChatGPT, Claude, Copilot, Gemini, Cursor, etc.) while
working on this assignment: which ones, how you used them, and whether they helped
or got in the way of understanding the material.

**Being honest here will not cost you points.** Writing `"none"` earns full marks,
and so does describing heavy use. We are trying to learn what actually helps people
learn, which only works if the reports are accurate.

In [ ]:
# Replace the empty string with your answer. Anything you write earns full marks --
# including "none". A few sentences is plenty.
AI_USE = ""


# PennGrader grading command
grader.grade('test_ai_use_report', AI_USE)

### **2. Time spent [2 pts]**

Roughly how long did this assignment take you, and how was that time split across
the parts? Rough numbers are genuinely useful — please do not agonize over precision.

You will get a far better estimate if you jot this down **as you go** rather than
reconstructing it at the end.

In [ ]:
# Replace the empty string with your answer. Anything you write earns full marks.
# For example: "about 7 hours -- 2 on K-means, 1 on PCA, 4 on the CNN".
TIME_SPENT_HOURS = ""


# PennGrader grading command
grader.grade('test_time_report', TIME_SPENT_HOURS)

# **Submit to Gradescope**

Congratulations on completing the HW3 coding assignment!

**Before you download — check your figures and answers are there.** Sections 1.4, 1.5, 1.6,
2.1, 2.2.2 and 2.3 are marked by hand from the plots and the written answers in this notebook.
Scroll back and make sure every plot is showing and every answer cell has your own words in it. A
notebook downloaded with its outputs cleared arrives with nothing to mark in those sections, and
they are 15 of the 50 points.

**Check that 3.5's grading cell ran in Colab** — Gradescope skips it when it re-runs your
notebook.

The final step is to download this iPython notebook and submit it to [Gradescope](https://www.gradescope.com).

*To download the notebook from Google Colab, click*
`File` → `Download` → `Download .ipynb`